# Simulation des résultats du sondage sur l'efficacité du Pass Culture

Dans ce notebook, nous allons simuler les résultats du sondage mené auprès des utilisateurs du Pass Culture afin d'en évaluer l'efficacité.

Les bases de données utilisées ci-après ont été récupérées auprès de la Cour des Comptes : https://www.ccomptes.fr/fr/publications/premier-bilan-du-pass-culture. 

Tout d'abord, nous devons impoorter les éléments nécessaires.

In [48]:
from functions import *
import numpy as np
import pandas as pd


## Importation des données

Dans un premier temps, nous importons les 16 bases de données mises à disposition par la Cour des Comptes. Elles regroupent des statistiques agrégées et nous pouvons ainsi reproduire les figures présentées dans le rapport d'évaluation de la Cour.

In [49]:
dfs = {}  # dictionnaire pour stocker tous les DataFrames
for elt in ["C3", "C4", "G1", "G2", "G3", "G4", "G5", "G6", "G7", "G8", "G9", "G10", "G11", "G13", "G14", "G15"]:  # liste des df que l'on veut importer
    path = elt+".csv"
    dfs[f"df_{elt.lower()}"] = pd.read_csv(path, sep=",", encoding="latin-1") 
    # On ne peut pas utiliser importdata car l'encodage du fichier n'est pas le même

Nous avons donc obtenu 16 bases de données qui reprennent des statistiques descriptives sur le sondage à partir duquel le dispositif a été évalué. Nous devons maintenant simuler les données à partir de ces statistiques agrégées.

## Description des données à notre disposition

Dans un premier temps, nous allons récapituler les données dont nous disposons, puis nous passerons à la simulation. Pour créer le tableau suivant, nous avons utilisé la fonction "infosbase" sur chaque dataframe. 

<table>
  <caption>
    Données mises à disposition par la Cour des Comptes
  </caption>
  <thead>
    <tr>
      <th scope="col">Nom du data_frame</th>
      <th scope="col">Variables</th>
      <th scope="col">Nombre de lignes</th>
      <th scope="col">Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th scope="row">df_c3</th>
      <td>Département<br>Montant.moyen.dépensé.par.les.jeunes.du.département</td>
      <td>102 lignes</td>
      <td>Donne le montant moyen dépensé par les jeunes du département.</td>
    </tr>
    <tr>
      <th scope="row">df_c4</th>
      <td>Département<br>Score.de.diversification.moyen.par.département</td>
      <td>102 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g1</th>
      <td>Âge<br>15.+<br>16.+<br>17.+<br>18.+</td>
      <td>1 ligne</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g2</th>
      <td>Caractéristique<br>Valeur<br>Taux.d'activitation.du.pass</td>
      <td>12 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g3</th>
      <td>Statut.déclaré<br>Etudiant<br>Lycéen<br>Collégien<br>Apprenti,.alternant,.service.civique<br>Demandeur.d'emploi<br>Employé<br>Inactif</td>
      <td>1 ligne</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g4</th>
      <td>Trimestre<br>cat_agrr<br>prop_aggr</td>
      <td>66 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g5</th>
      <td>categories<br>Population<br>Pourcentage</td>
      <td>20 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g6</th>
      <td>trimestre<br>macro_rayon_r<br>prop_montant</td>
      <td>99 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g7</th>
      <td>categories<br>nombre_utilisateurs<br>Revenu</td>
      <td>42 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g8</th>
      <td>trimestre<br>Age.à.la.réservation<br>home<br>search</td>
      <td>20 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g9</th>
      <td>Origin<br>Composante.diversité<br>Prop</td>
      <td>10 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g10</th>
      <td>Origine<br>Catégorie<br>prop</td>
      <td>12 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g11</th>
      <td>Delta.de.diversification<br>Part.des.réservations</td>
      <td>6 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g13</th>
      <td>X1<br>LFI.2022<br>Exec.2022<br>LFI.2023<br>Exec.2023<br>LFI.2024<br>Exec.(prév.).2024</td>
      <td>2 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g14</th>
      <td>Service<br>Effectif</td>
      <td>6 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g15</th>
      <td>X1<br>16.ans<br>17.ans<br>18.ans<br>19.ans</td>
      <td>4 lignes</td>
      <td></td>
    </tr>
  </tbody>
</table>


In [50]:
infosbase(dfs["df_c3"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 102
Nombre de colonnes  : 2

NOMS DES COLONNES ET TYPES
Départements                                            object
Montant.moyen.dépensé.par.les.jeunes.du.département    float64
dtype: object


In [51]:
print(dfs["df_c3"])

    Départements  Montant.moyen.dépensé.par.les.jeunes.du.département
0             01                                              238.0  
1             02                                              238.0  
2             03                                              238.0  
3             04                                              230.0  
4             05                                              236.0  
..           ...                                                ...  
97           972                                              249.0  
98           973                                              238.0  
99           974                                              224.0  
100          975                                              215.0  
101          976                                              184.0  

[102 rows x 2 columns]


In [52]:
infosbase(dfs["df_c4"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 102
Nombre de colonnes  : 2

NOMS DES COLONNES ET TYPES
Départements                                       object
Score.de.diversification.moyen.par.département    float64
dtype: object


In [53]:
print(dfs["df_c4"])

    Départements  Score.de.diversification.moyen.par.département
0             01                                       11.916181
1             02                                       11.696677
2             03                                       11.521567
3             04                                       11.499613
4             05                                       13.224819
..           ...                                             ...
97           972                                       12.423631
98           973                                       13.707500
99           974                                       10.882557
100          975                                       15.000000
101          976                                       12.791667

[102 rows x 2 columns]


In [54]:
infosbase(dfs["df_g1"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 1
Nombre de colonnes  : 5

NOMS DES COLONNES ET TYPES
Âge      object
15.+    float64
16.+    float64
17.+    float64
18.+    float64
dtype: object


In [55]:
print(dfs["df_g1"])

                  Âge  15.+  16.+  17.+  18.+
0  Taux de couverture  0.48  0.62  0.74  0.82


In [56]:
infosbase(dfs["df_g2"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 12
Nombre de colonnes  : 3

NOMS DES COLONNES ET TYPES
Caractéristique                 object
Valeur                          object
Taux.d'activitation.du.pass    float64
dtype: object


In [57]:
print(dfs["df_g2"])

              Caractéristique                  Valeur  \
0                        Sexe                   Femme   
1                        Sexe                   Homme   
2                   Situation  Travailleur ou Inactif   
3                   Situation                Etudiant   
4             Origine sociale               Populaire   
5             Origine sociale                 Moyenne   
6             Origine sociale              Supérieure   
7   Taille de l'agglomération                     <2k   
8   Taille de l'agglomération                  2k-20k   
9   Taille de l'agglomération                20k-100k   
10  Taille de l'agglomération                   >100k   
11  Taille de l'agglomération                   Paris   

    Taux.d'activitation.du.pass  
0                      0.792977  
1                      0.691525  
2                      0.656367  
3                      0.812401  
4                      0.679880  
5                      0.735280  
6                   

In [58]:
infosbase(dfs["df_g3"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 1
Nombre de colonnes  : 8

NOMS DES COLONNES ET TYPES
Statut.déclaré                          object
Etudiant                                 int64
Lycéen                                   int64
Collégien                                int64
Apprenti,.alternant,.service.civique     int64
Demandeur.d'emploi                       int64
Employé                                  int64
Inactif                                  int64
dtype: object


In [59]:
print(dfs["df_g3"])

  Statut.déclaré  Etudiant   Lycéen  Collégien  \
0         Nombre   1156653  2224717     249732   

   Apprenti,.alternant,.service.civique  Demandeur.d'emploi  Employé  Inactif  
0                                242931              108120    55865    21927  


In [60]:
infosbase(dfs["df_g4"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 66
Nombre de colonnes  : 3

NOMS DES COLONNES ET TYPES
Trimestre      int64
cat_agrr      object
prop_aggr    float64
dtype: object


In [61]:
print(dfs["df_g4"])

    Trimestre          cat_agrr  prop_aggr
0       44378             Autre   0.034625
1       44378  Spectacle vivant   0.012120
2       44378         Art&Musée   0.017104
3       44378           Musique   0.182411
4       44378       Audiovisuel   0.201918
..        ...               ...        ...
61      45292  Spectacle vivant   0.015344
62      45292         Art&Musée   0.018794
63      45292           Musique   0.277502
64      45292       Audiovisuel   0.208467
65      45292             Livre   0.427380

[66 rows x 3 columns]


In [62]:
infosbase(dfs["df_g5"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 20
Nombre de colonnes  : 3

NOMS DES COLONNES ET TYPES
categories      object
Population      object
Pourcentage    float64
dtype: object


In [63]:
print(dfs["df_g5"])

             categories   Population  Pourcentage
0                 LIVRE  Echantillon    72.753208
1                 LIVRE   15-17 ans     39.796031
2                CINEMA   15-17 ans     30.966524
3                CINEMA  Echantillon    50.406285
4          MUSIQUE_LIVE  Echantillon    20.167240
5           AUDIOVISUEL  Echantillon    19.707559
6   MUSIQUE_ENREGISTREE  Echantillon    18.006155
7   PRATIQUE ARTISTIQUE  Echantillon    14.028219
8           AUDIOVISUEL   15-17 ans      7.090551
9   MUSIQUE_ENREGISTREE   15-17 ans      5.798373
10                  JEU  Echantillon     8.085119
11            SPECTACLE  Echantillon     7.771647
12                MUSEE  Echantillon     6.175865
13         MUSIQUE_LIVE   15-17 ans      3.432501
14               AUTRES  Echantillon     4.495398
15                MUSEE   15-17 ans      2.267777
16               AUTRES   15-17 ans      1.994552
17  PRATIQUE ARTISTIQUE   15-17 ans      1.875672
18                  JEU   15-17 ans      0.919760


In [64]:
infosbase(dfs["df_g6"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 99
Nombre de colonnes  : 3

NOMS DES COLONNES ET TYPES
trimestre          int64
macro_rayon_r     object
prop_montant     float64
dtype: object


In [65]:
print(dfs["df_g6"])

    trimestre              macro_rayon_r  prop_montant
0       44378                       Arts      0.025046
1       44378                     Autres      0.156262
2       44378           Bandes dessinées      0.055262
3       44378          Sciences sociales      0.180702
4       44378                   Jeunesse      0.074487
..        ...                        ...           ...
94      45292                   Jeunesse      0.081761
95      45292                Littérature      0.309268
96      45292                      Manga      0.207225
97      45292           Poèsie & théâtre      0.016123
98      45292  Religions, spiritualitées      0.021884

[99 rows x 3 columns]


In [66]:
infosbase(dfs["df_g7"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 42
Nombre de colonnes  : 3

NOMS DES COLONNES ET TYPES
categories             object
nombre_utilisateurs     int64
Revenu                 object
dtype: object


In [67]:
print(dfs["df_g7"])

             categories  nombre_utilisateurs  \
0                 LIVRE                62773   
1                CINEMA                41362   
2          MUSIQUE_LIVE                16476   
3                  FILM                16195   
4   MUSIQUE_ENREGISTREE                13586   
5             SPECTACLE                12598   
6                 MUSEE                 8038   
7                   JEU                 7571   
8            INSTRUMENT                 5952   
9            BEAUX_ARTS                 5183   
10                MEDIA                 4220   
11           CONFERENCE                 1200   
12         PRATIQUE_ART                 1172   
13         CARTE_JEUNES                   19   
14                LIVRE               123572   
15               CINEMA                82126   
16         MUSIQUE_LIVE                36888   
17                 FILM                33952   
18  MUSIQUE_ENREGISTREE                32272   
19                  JEU                1

In [68]:
infosbase(dfs["df_g8"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 20
Nombre de colonnes  : 4

NOMS DES COLONNES ET TYPES
trimestre                 int64
Age.à.la.réservation     object
home                    float64
search                  float64
dtype: object


In [69]:
print(dfs["df_g8"])

    trimestre Age.à.la.réservation      home    search
0       44378            18-20 ans  0.025803  0.751042
1       44470            18-20 ans  0.031239  0.736710
2       44562            15-17 ans  0.071523  0.676753
3       44562            18-20 ans  0.027475  0.740153
4       44652            15-17 ans  0.057275  0.663266
5       44652            18-20 ans  0.027478  0.730922
6       44743            15-17 ans  0.056394  0.661101
7       44743            18-20 ans  0.019723  0.726999
8       44835            15-17 ans  0.078845  0.615005
9       44835            18-20 ans  0.029206  0.657485
10      44927            15-17 ans  0.125247  0.588801
11      44927            18-20 ans  0.047104  0.661242
12      45017            15-17 ans  0.122548  0.581462
13      45017            18-20 ans  0.042608  0.641855
14      45108            15-17 ans  0.129735  0.567015
15      45108            18-20 ans  0.043007  0.638144
16      45200            15-17 ans  0.155471  0.511102
17      45

In [70]:
infosbase(dfs["df_g9"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 10
Nombre de colonnes  : 3

NOMS DES COLONNES ET TYPES
Origin                   object
Composante.diversité     object
Prop                    float64
dtype: object


In [71]:
print(dfs["df_g9"])

           Origin Composante.diversité      Prop
0  Page d'accueil            Catégorie  0.306116
1  Page d'accueil       Sous-catégorie  0.406772
2  Page d'accueil               Format  0.207298
3  Page d'accueil                Lieux  0.367152
4  Page d'accueil                Genre  0.530976
5       Recherche            Catégorie  0.099952
6       Recherche       Sous-catégorie  0.123400
7       Recherche               Format  0.059102
8       Recherche                Lieux  0.221954
9       Recherche                Genre  0.322869


In [72]:
infosbase(dfs["df_g10"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 12
Nombre de colonnes  : 3

NOMS DES COLONNES ET TYPES
Origine       object
Catégorie     object
prop         float64
dtype: object


In [73]:
print(dfs["df_g10"])

           Origine         Catégorie      prop
0   Page d'accueil             Livre  0.177928
1   Page d'accueil             Autre  0.094550
2   Page d'accueil       Audiovisuel  0.565626
3   Page d'accueil           Musique  0.122686
4   Page d'accueil  Spectacle vivant  0.015293
5   Page d'accueil         Art&Musée  0.023916
6        Recherche             Livre  0.716512
7        Recherche             Autre  0.040126
8        Recherche       Audiovisuel  0.139815
9        Recherche           Musique  0.075914
10       Recherche  Spectacle vivant  0.005989
11       Recherche         Art&Musée  0.021643


In [74]:
infosbase(dfs["df_g11"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 6
Nombre de colonnes  : 2

NOMS DES COLONNES ET TYPES
Delta.de.diversification      int64
Part.des.réservations       float64
dtype: object


In [75]:
print(dfs["df_g11"])

   Delta.de.diversification  Part.des.réservations
0                         0                 0.5246
1                         1                 0.2012
2                         2                 0.1128
3                         3                 0.0250
4                         4                 0.0702
5                         5                 0.0663


In [76]:
infosbase(dfs["df_g13"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 2
Nombre de colonnes  : 7

NOMS DES COLONNES ET TYPES
X1                    object
LFI.2022               int64
Exec.2022            float64
LFI.2023             float64
Exec.2023            float64
LFI.2024             float64
Exec.(prév.).2024    float64
dtype: object


In [77]:
print(dfs["df_g13"])

                  X1  LFI.2022  Exec.2022  LFI.2023  Exec.2023  LFI.2024  \
0  Part individuelle       199      199.6     209.5      240.1     210.5   
1    Part collective        45       18.0      51.0       51.0      62.0   

   Exec.(prév.).2024  
0              244.4  
1               80.2  


In [78]:
infosbase(dfs["df_g14"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 6
Nombre de colonnes  : 2

NOMS DES COLONNES ET TYPES
Service     object
Effectif     int64
dtype: object


In [79]:
print(dfs["df_g14"])

                                           Service  Effectif
0                              Direction technique        46
1                                               SG        19
2  Direction du pilotage (Direction des opération)        26
3                    Direction de la communication        17
4                                Direction produit        17
5                       Direction du développement        51


In [80]:
infosbase(dfs["df_g15"])

DIMENSIONS DU JEU DE DONNÉES
Nombre de lignes    : 4
Nombre de colonnes  : 5

NOMS DES COLONNES ET TYPES
X1          int64
16.ans    float64
17.ans    float64
18.ans    float64
19.ans    float64
dtype: object


In [81]:
print(dfs["df_g15"])

     X1  16.ans  17.ans  18.ans  19.ans
0  2020     NaN     NaN    0.00    0.02
1  2021     NaN     NaN    0.09    0.34
2  2022    0.19    0.25    0.39    0.49
3  2023    0.33    0.45    0.57    0.63


## Simulation 

Objectif :

Simuler une base de données au niveau individuel à partir de statistiques agrégées (moyennes, proportions, tableaux croisés)
publiées dans un rapport (ici : Cour des comptes, Pass Culture).
 
Structure du script :

1. Fondements théoriques : explication de la théorie mathématique sur laquelle se fonde la simulation, justification du choix de la méthode ;

2. Configuration générale (graine aléatoire, taille de l'échantillon)

3. Fonction d'ajustement proportionnel itératif (IPF / raking)
       -> pour caler une table jointe sur plusieurs marges connues

4. Fonction de validation : on ré-agrège les données simulées et on
       les compare aux statistiques d'origine

Les fonctions à créer seront à nouveaux définies dans le fichier functions.py afin de fluidifier la lecture du notebook.

### Fondements théoriques

#### Problème : inférence écologique

La désagrégation statistique répond à un problème d'inférence statistique, ou inférence écologique. On dispose de données agrégées (moyennes, pourcentages, totaux par groupes), et on souhaite reconstituer une base de données au niveau individuel, alors qu'on ne dispose pas des données à ce niveau.

L'objectif n'est ainsi pas de retrouver les "vraies" données, mais de simuler une base de données qui aurait mené aux mêmes résultats agrégés. Nous sommes donc bien dans une situation de simulation sous contrainte, et non de reconstruction.

L'inférence est ici dite "stochastique car la désagrégation repose sur des tirages aléatoires (type Monte Carlo <mark>à développer / préciser</mark>), plutôt que sur une règle déterministe. La valeur prise par une certaine variable est tirée aléatoirement pour chaque individu, afin de conserver l'aléa et la variabilité naturels que l'on observerait dans un véritable échantillon.

#### Difficultés principales 

La difficulté principale pour notre simulation est que les lois marginales ne permettent pas de déduire la loi jointe. 

Dans certains cas, l'hypothèse la plus simple est de supposer l'indépendance des variables. Chaque variable sera alors simulée séparément, indépendamment des autres. Cette solution est rapide est simple, mais elle peut introduire un biais lorsqu'en réalité, les variables sont corrélées.

Lorsque nous disposons de tableaux qui croisent déjà plusieurs variables, nous pourrons nous passer de l'hypothèse d'indépendance. En effet, nous avons alors des informations sur la loi jointe des différentes variables proposées, et nous pourrons donc les simuler ensemble au lieu de les simuler indépendamment. Cette option sera préférable lorsqu'elle est possible.

Ces deux options sont les deux pôles à partir desquels nous allons effectuer la simulation. 


### Configuration générale

La configuration est une étape importante car elle garantit la reproductibilité des résultats.

In [82]:
seed = 42             # graine aléatoire -> reproductibilité des simulations
nb_indiv = 10000      # taille de l'échantillon simulé (à ajuster)
 
rng = np.random.default_rng(seed)

### Fonction d'ajustement proportionnel

L'ajustement proportionnel itératif (*Iterative Proportional Fitting*, IPF), aussi appelé **raking**, 
permet de construire une table jointe compatible avec plusieurs lois marginales connues, sans jamais 
observer directement la loi jointe.

**Principe :**

1. On part d'une table initiale (par exemple sous hypothèse d'indépendance : le produit des marges normalisées) ;
2. À chaque itération, pour chaque variable à caler, on recalcule la marge courante de la table simulée et on la compare à la marge cible (issue des tableaux de la Cour des comptes) ;
3. On multiplie chaque ligne par le ratio (marge cible / marge courante) correspondant à sa modalité ;
4. On répète pour toutes les variables jusqu'à convergence (écart maximal entre marges courantes et marges cibles inférieur à un seuil `tol`).

Cet algorithme converge (sous des conditions assez générales) vers une table qui respecte toutes les marges fournies, tout en restant "la plus proche possible" de la table initiale au sens de la divergence de Kullback-Leibler. C'est la méthode standard de calage sur marges utilisée en statistique officielle (post-stratification, calage d'enquêtes).

### Validation

Une fois la base individuelle simulée, il est indispensable de vérifier qu'elle reproduit bien les 
statistiques agrégées de départ : c'est la seule façon de s'assurer que la simulation n'a pas introduit de 
biais. Pour cela, on ré-agrège les données simulées (calcul des proportions ou effectifs par modalité) et on 
compare ces valeurs aux marges cibles extraites des tableaux de la Cour des comptes.